# Zadanie 3: optymalizacja dyskretna

Termin realizacji: 20 kwietnia 2026

Zadanie do oddania przez MS Teams. Do oddania: kod oraz krótkie sprawozdanie w PDF (można na przykład przy użyciu `quarto render notebook.ipynb --to pdf`).

## Na 3.0

Do realizacji:

1. Zaimplementuj dyskretny problem plecakowy z trzema plecakami w MiniZinc na podstawie przykładu (plik `minizinc.ipynb`). Spróbuj rozwiązać problem dla 10 zestawów parametrów o różnych wielkościach tak, aby rozwiązanie największego problemu trwało powyżej 5 sekund. Zanotuj w każdym przypadku liczbę wszystkich przedmiotów, pojemności plecaków, liczbę wybranych przedmiotów i sumaryczną wartość przedmiotów w każdym plecaku osobno.
2. Losowanie przedmiotów w każdym przypadku powinno być tak ustawione, aby optymalne rozwiązanie wymagało wzięcia przynajmniej dwóch przedmiotów do każdego plecaka oraz zostawienia przynajmniej dwóch przedmiotów poza plecakami. W raporcie należy umieścić informację w jaki sposób zostało to zweryfikowane.
3. Zmodyfikuj metodę z notatnika `tabu_search.ipynb` tak aby rozwiązywała opisywany problem plecakowy. Porównaj na tych samych problemach czy Minizinc i Tabu search zwracają równie dobre rozwiazania, oraz wypisz jakie to są rozwiązania. Wykonaj eksperymenty z trzema różnymi długościami listy zakazów (1, 2, 5).

## Na 4.0

Do realizacji:

1. Punkty z zadania na 3.0.
2. Rozszerz możliwe ruchy w tabu search o przeniesienie przedmiotu z jednego plecaka do drugiego. Zapisz rozważ czy to poprawia działanie metody (czy znalezione jest lepsze, takie samo czy gorsze rozwiązanie? czy rozwiązanie jest znajdowane szybciej czy wolniej?). Dla każdego z 10 zestawów parametrów problemu plecakowego wykonaj ocenę przez uśrednienie dla 10 różnych losowych przypadków.
3. Podsumuj dane w formie tabelki z czterema kolumnami (Minizinc, tabu search z listą o długości 1, 2, i 5) oraz 10 wierszami (po jednym dla zestawu parametrów problemu), a w komórkach umieść średnią wartość wartości przedmiotów oraz średni czas potrzebny do uzyskania rozwiązania.

## Na 5.0

Do realizacji:

1. Punkty z zadania na 4.0.
2. Zaimplementuj samodzielnie algorytm symulowanego wyżarzania z podobnym interfejsem co tabu search. Porównaj jego działanie do rozważanych wcześniej rozwiązań dla trzech różnych schematów chłodzenia. Dobierz liczbę iteracji tak, aby czas działania był porównywalny do tabu search.


In [1]:
import Pkg
Pkg.add("Distributions")
Pkg.add("JuMP")
Pkg.add("MiniZinc")
Pkg.add("MathOptInterface")
Pkg.add("DataStructures")
Pkg.add("Distributions")

   Resolving package versions...
     Project No packages added to or removed from `~/.julia/environments/v1.12/Project.toml`
    Manifest No packages added to or removed from `~/.julia/environments/v1.12/Manifest.toml`
   Resolving package versions...
     Project No packages added to or removed from `~/.julia/environments/v1.12/Project.toml`
    Manifest No packages added to or removed from `~/.julia/environments/v1.12/Manifest.toml`
   Resolving package versions...
     Project No packages added to or removed from `~/.julia/environments/v1.12/Project.toml`
    Manifest No packages added to or removed from `~/.julia/environments/v1.12/Manifest.toml`
   Resolving package versions...
     Project No packages added to or removed from `~/.julia/environments/v1.12/Project.toml`
    Manifest No packages added to or removed from `~/.julia/environments/v1.12/Manifest.toml`
   Resolving package versions...
     Project No packages added to or removed from `~/.julia/environments/v1.12/Project.

## Generowanie plików z problemami

In [2]:
using Distributions

function make_dzn(n::Int, id::Int, R::Int = 100, α::Float64 = 0.5)
    weights = rand(DiscreteUniform(R / 10, R), n)

    # Strong corelation between profits and weights makes dataset hard
    profits = [rand(DiscreteUniform(R / 10, weight)) + R / 10 for weight in weights]
    profits = Array{Int}(profits)

    # capacity is typically some α times sum of weights
    capacity = Int64(round(α * sum(weights)));
    content = """
    ITEM = _(1..$n);
    capacity = $capacity;
    profits = $profits;
    weights = $weights;
    """
    file = open("knapsack_$id.dzn", "w+")
    write(file, content)
    close(file)
end

make_dzn (generic function with 3 methods)

In [42]:
n_items = [5, 5, 5, 10, 15, 40]

for (id, i) in enumerate(n_items)
    make_dzn(i, id)
end

In [43]:
# Gecode to g..no (używaj cp-sat albo highsa)

for i in 1:length(n_items)
    result = read(`minizinc --solver cp-sat my_knapsack.mzn knapsack_$i.dzn` , String)
    println(result)
end

taken = [false, false, true, true, false];
profit = 68;
weight = 121;
----------

taken = [true, false, false, false, true];
profit = 87;
weight = 73;
----------

taken = [true, false, false, true, true];
profit = 102;
weight = 138;
----------

taken = [false, false, false, false, true, true, true, true, false, true];
profit = 267;
weight = 273;
----------

taken = [false, true, true, false, false, true, true, true, false, true, false, true, true, true, false];
profit = 353;
weight = 351;
----------

taken = [true, true, false, false, false, true, false, true, true, false, true, false, false, false, true, true, true, false, false, false, false, true, false, true, true, false, true, false, false, true, true, true, true, true, true, true, true, true, true, true];
profit = 1063;
weight = 1127;
----------



## Tabu Search

In [51]:
using DataStructures
using Distributions

mutable struct TabuState{TMove,P,TF}
    tabu_buffer::CircularBuffer{TMove}
    best_seen::P
    best_seen_obj::TF
    current::P
    considered::P
    iter::Int
end

function TabuState(p, x0; buffer_length::Int=10)
    moves = possible_moves(p, x0)
    obj = objective(p, x0)
    return TabuState{eltype(moves),typeof(x0),typeof(obj)}(
        CircularBuffer{eltype(moves)}(buffer_length), x0, obj, copy(x0), copy(x0), 1
    )
end


function solve_tabu(p, s::TabuState; iteration_limit::Int=100)
    while s.iter < iteration_limit
        moves = possible_moves(p, s.current)
        best_move = 0
        best_move_obj = Inf
        for (i_move, move) in enumerate(moves)
            if in(move, s.tabu_buffer)
                # move forbidden, do not consider
                continue
            end
            # evaluate move
            copyto!(s.considered, s.current)
            apply!(s.considered, move)
            considered_value = objective(p, s.considered)
            if considered_value < best_move_obj
                best_move = i_move
                best_move_obj = considered_value
            end
        end
        # no allowed move found
        if best_move == 0
            break
        end
        apply!(s.current, moves[best_move])
        push!(s.tabu_buffer, invert_move(p, moves[best_move]))
        if best_move_obj < s.best_seen_obj
            # best so far, let's remember it
            copyto!(s.best_seen, s.current)
            s.best_seen_obj = best_move_obj
        end
        s.iter += 1
    end
    return s.best_seen
end


struct KnapsackProblem
    capacity::Int
    weights::Vector{Int}
    profits::Vector{Int}
end

function objective(p::KnapsackProblem, x)
    return -sum(p.profits .* x)
end


function apply!(x, move::Tuple{Symbol,Int})
    if move[1] === :add
        x[move[2]] = true
    else
        x[move[2]] = false
    end
    return x
end

function invert_move(::KnapsackProblem, move::Tuple{Symbol,Int})
    if move[1] === :add
        return (:remove, move[2])
    else
        return (:add, move[2])
    end
end


function possible_moves(p::KnapsackProblem, x::Vector{Bool})
    move_list = Tuple{Symbol,Int}[]
    current_weight = sum(p.weights .* x)
    # add item
    for i in eachindex(x, p.weights)
        if !x[i] && current_weight + p.weights[i] <= p.capacity
            push!(move_list, (:add, i))
        end
    end
    # remove item
    for i in eachindex(x, p.weights)
        if x[i]
            push!(move_list, (:remove, i))
        end
    end
    return move_list
end



possible_moves (generic function with 1 method)

In [52]:
function read_problem(file_path::String)
    data = Dict{Symbol, Any}()
    
    open(file_path, "r") do file
        for line in eachline(file)
            clean_line = strip(line)
            clean_line = replace(clean_line, ";" => "")
            
            if isempty(clean_line) || startswith(clean_line, "%")
                continue
            end

            parts = split(clean_line, "=", limit=2)
            if length(parts) == 2
                var_name = Symbol(strip(parts[1]))
                var_value_str = strip(parts[2])

                if occursin("..", var_value_str)
                    m = match(r"(\d+)\.\.(\d+)", var_value_str)
                    if m !== nothing
                        start_val = parse(Int, m.captures[1])
                        end_val = parse(Int, m.captures[2])
                        data[var_name] = start_val:end_val
                    end
                else
                    try
                        data[var_name] = eval(Meta.parse(var_value_str))
                    catch e
                        @warn "Could not parse value for $var_name: $var_value_str"
                    end
                end
            end
        end
    end
    
    return KnapsackProblem(data[:capacity], data[:weights], data[:profits])
end

read_problem (generic function with 1 method)

In [53]:
for i in 1:length(n_items)
    result = read(`minizinc --solver cp-sat my_knapsack.mzn knapsack_$i.dzn` , String)
    println("Minizinc:")
    println(result)

    kp = read_problem("knapsack_$i.dzn")

    x0 = fill(false, length(kp.weights))
    st = TabuState(kp, x0; buffer_length=10)
    sol = solve_tabu(kp, st; iteration_limit=1000000)

    println("Tabu Search:")
    println(sol.==1)
    println("Profit: ", -st.best_seen_obj)
    #println("Last iteration: ", st.iter)

    println("---------")
    println("=========")
end

Minizinc:
taken = [false, false, true, true, false];
profit = 68;
weight = 121;
----------

Tabu Search:
Bool[0, 0, 1, 1, 0]
Profit: 68
---------
Minizinc:
taken = [true, false, false, false, true];
profit = 87;
weight = 73;
----------

Tabu Search:
Bool[1, 0, 0, 1, 0]
Profit: 87
---------
Minizinc:
taken = [true, false, false, true, true];
profit = 102;
weight = 138;
----------

Tabu Search:
Bool[0, 0, 1, 0, 1]
Profit: 100
---------
Minizinc:
taken = [false, false, false, false, true, true, true, true, false, true];
profit = 267;
weight = 273;
----------

Tabu Search:
Bool[0, 0, 1, 0, 1, 0, 1, 0, 0, 0]
Profit: 227
---------
Minizinc:
taken = [false, true, true, false, false, true, true, true, false, true, false, true, true, true, false];
profit = 353;
weight = 351;
----------

Tabu Search:
Bool[0, 0, 1, 0, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0]
Profit: 318
---------
Minizinc:
taken = [true, true, false, false, false, true, false, true, true, false, true, false, false, false, true, true, tru

## Symulowane Wyżarzanie

In [54]:
using DataStructures
using Distributions

mutable struct SAState{P, TF}
    best_seen::P
    best_seen_obj::TF
    current::P
    current_obj::TF
    considered::P
    temp::Float64
    iter::Int
end

function SAState(p, x0; initial_temp::Float64=100.0)
    obj = objective(p, x0)
    return SAState{typeof(x0), typeof(obj)}(
        copy(x0), obj, copy(x0), obj, copy(x0), initial_temp, 1
    )
end

function solve_sa(p, s::SAState; iteration_limit::Int=1000, cooling_rate::Float64=0.95)
    while s.iter < iteration_limit
        move = random_move(p, s.current) 
        
        copyto!(s.considered, s.current)
        apply!(s.considered, move)
        trial_obj = objective(p, s.considered)
    
        ΔE = trial_obj - s.current_obj
        
        if ΔE < 0 || rand() < exp(-ΔE / s.temp)
            copyto!(s.current, s.considered)
            s.current_obj = trial_obj
            
            if s.current_obj < s.best_seen_obj
                copyto!(s.best_seen, s.current)
                s.best_seen_obj = s.current_obj
            end
        end
        
        s.temp *= cooling_rate
        s.iter += 1
    end
    return s.best_seen
end


struct KnapsackProblem
    capacity::Int
    weights::Vector{Int}
    profits::Vector{Int}
end

function random_move(p::KnapsackProblem, x::Vector{Bool})
    i = rand(1:length(x))
    
    current_weight = sum(p.weights .* x)
    
    if x[i]
        return (:remove, i)
    else
        if current_weight + p.weights[i] <= p.capacity
            return (:add, i)
        else
            in_bag_indices = findall(x)
            if isempty(in_bag_indices)
                return (:add, i)
            end
            return (:remove, rand(in_bag_indices))
        end
    end
end

function objective(p::KnapsackProblem, x)
    return -sum(p.profits .* x)
end


function apply!(x, move::Tuple{Symbol,Int})
    if move[1] === :add
        x[move[2]] = true
    else
        x[move[2]] = false
    end
    return x
end



apply! (generic function with 1 method)

In [55]:
function generate_problem()
    n_items = 100
    profits = rand(DiscreteUniform(10, 1000), n_items)
    weights = rand(DiscreteUniform(10, 100), n_items)
    kp = KnapsackProblem(3000, profits, weights)
end

kp1 = generate_problem()


KnapsackProblem(3000, [419, 154, 693, 376, 854, 359, 631, 37, 964, 762  …  849, 373, 548, 863, 744, 594, 332, 20, 230, 686], [53, 90, 55, 53, 41, 28, 73, 28, 23, 47  …  33, 24, 60, 57, 17, 99, 95, 95, 49, 80])

In [56]:
function test_sa(kp)
    x0 = fill(false, length(kp.weights))
    
    st = SAState(kp, x0; initial_temp=1000.0) 
    
    sol = solve_sa(kp, st; iteration_limit=1000000, cooling_rate=0.99999)
    
    println("Items selected (indices): ", findall(sol))
    println("Best objective (Profit): ", -st.best_seen_obj)
    println("Total Weight: ", sum(kp.weights[sol]))
    println("Last iteration: ", st.iter)
end

test_sa (generic function with 1 method)

In [57]:
test_sa(kp1)

Items selected (indices): [2, 8, 13, 22, 24, 32, 36, 45, 48, 60, 62, 68, 69, 70, 74, 81, 82, 84, 97, 98]
Best objective (Profit): 1507
Total Weight: 2978
Last iteration: 1000000
